In [1]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\huggingface-text-summarization\\notebooks'

In [2]:
import os
os.chdir("../")
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\huggingface-text-summarization'

In [8]:
from dataclasses import dataclass
from pathlib import Path

from src.huggingface_text_summarization.constants import *
from src.huggingface_text_summarization.utils.common import read_yaml, create_directories

import torch
import pandas as pd
import evaluate
from tqdm import tqdm
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [4]:
@dataclass
class ModelEvaluationConfig:
    root_directory: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH, param_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(path_to_yaml=config_file_path)
        self.params = read_yaml(path_to_yaml=param_file_path)

        create_directories(path_to_directories=[self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories(path_to_directories=[config.root_directory])

        model_evaluation_config = ModelEvaluationConfig(
            root_directory=config.root_directory,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name,
        )

        return model_evaluation_config


In [10]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """
        Split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements.
        """
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self,
        dataset, metric, model, tokenizer,
        batch_size=16, device="cuda",
        column_text="article", column_summary="highlights"
        ):

        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):

            inputs = tokenizer(
                article_batch, max_length=1024,  truncation=True,
                padding="max_length", return_tensors="pt"
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8, num_beams=8, max_length=128
            )
            ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''

            # Finally, we decode the generated texts,
            # replace the  token, and add the decoded texts with the references to the metric.
            decoded_summaries = [
                tokenizer.decode(summary, skip_special_tokens=True,
                                clean_up_tokenization_spaces=True
                ) for summary in summaries
            ]

            decoded_summaries = [
                decoded_summary.replace("", " ") for decoded_summary in decoded_summaries]

            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        score = metric.compute()
        return score
    
    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path
        )
        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        ).to(device)

        # Loading Data
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

        rouge_metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum_pt['test'][0:10], metric=rouge_metric,
            model=model, tokenizer=tokenizer, batch_size=2, device=device,
            column_text='dialogue', column_summary='summary'
        )

        rouge_dict = {rouge_name: score[rouge_name] for rouge_name in rouge_names}

        df = pd.DataFrame(rouge_dict, index=[f'pegasus'])
        df.to_csv(self.config.metric_file_name, index=False, header=True)

In [11]:
config = ConfigurationManager()

model_evaluation_config = config.get_model_evaluation_config()
model_evaluation = ModelEvaluation(config=model_evaluation_config)

model_evaluation.evaluate()

[2026-03-21 01:24:49,557: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-21 01:24:49,568: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-21 01:24:49,569: INFO: common: created directory at: artifacts]
[2026-03-21 01:24:49,570: INFO: common: created directory at: artifacts/model_evaluation]


Loading weights: 100%|██████████| 682/682 [00:00<00:00, 4190.90it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
100%|██████████| 5/5 [03:39<00:00, 43.80s/it]

[2026-03-21 01:28:34,950: INFO: rouge_scorer: Using default tokenizer.]
